# Exploring the TTS Workspace with `tts-ctx`

This notebook demonstrates how to use `tts-ctx`, a progressive context discovery tool designed to help LLMs and developers navigate the complex multi-repo architecture of the Teamtools Studio (TTS) workspace.

## Core Objectives
- **Minimize Token Usage**: Instead of dumping whole files, `tts-ctx` provides 'progressive disclosure' (Depth 1 $\rightarrow$ 2 $\rightarrow$ 3).
- **Cross-Repo Visibility**: Resolve inheritance (MRO) and dependencies across ~190 independent repositories.
- **Consistency**: Standardize how agents find issue trackers, ownership, and domain context.

## 1. Indexing the Workspace
Before using the tool, you must build the symbol index. This parses all Python files in the workspace and resolves class hierarchies.

In [1]:
# Build the SQLite index and CSV dumps
!tts-ctx index

Indexing workspace at /Users/muszynsk/projects/tt_studio/dev...
  ⚠ sunrise-top: Missing description in pyproject.toml (or overrides/AGENTS.md)
  ⚬ sunrise-top: no src/ dir
  ⚠ Access-Key-Generation: Missing description in pyproject.toml (or overrides/AGENTS.md)
  ⚬ Access-Key-Generation: no src/ dir
  ⚠ CTD-Gen: Missing description in pyproject.toml (or overrides/AGENTS.md)
  ⚬ CTD-Gen: no src/ dir
  ⚠ Documentation: Missing description in pyproject.toml (or overrides/AGENTS.md)
  ⚬ Documentation: no src/ dir
  ⚠ File_Downlink_Corruption_Analysis: Missing description in pyproject.toml (or overrides/AGENTS.md)
  ⚬ File_Downlink_Corruption_Analysis: no src/ dir
  ⚠ credss: Missing description in pyproject.toml (or overrides/AGENTS.md)
  ⚬ credss: no src/ dir
  ⚠ development-environment: Missing description in pyproject.toml (or overrides/AGENTS.md)
  ⚬ development-environment: no src/ dir
  ⚠ ldap-acl: Missing description in pyproject.toml (or overrides/AGENTS.md)
  ⚬ ldap-acl: no src/ 

## 2. High-Level Orientation
The `start_here` command is the intended entry point for any agent starting a task. It provides a high-level map of the workspace and navigation suggestions.

In [2]:
!tts-ctx start-here

# tts-ctx — Where to start

# tts-ctx — Where to start

The TTS (Teamtools Studio) workspace is a collection of ~24 core libraries under
`tts_core/` and ~70 mission adaptation repos grouped by mission (`demosat/`,
`oco2/`, `fss/`, etc.). Each repo is an independent Git repository with its own
`AGENTS.md` and `docs/agents/README.md` pointer.

## Navigation strategy

1. **`tts-ctx core`** — see the shared libraries (data_utils, tower, events, …)
2. **`tts-ctx adaptations`** — see how missions extend core libraries
3. **`tts-ctx class <dotted.path>`** — dive into a specific class and its MRO
4. **`tts-ctx search <query>`** — fuzzy search any symbol across all repos
5. **`tts-ctx repo <repo-name>`** — get key classes + dependencies for one repo

## Key relationships

- Core libraries depend on `tts_utilities` (logger, Python compat shims).
- `tts_data_utils` → `tts_papertrail` → `tts_html_utils` → `tts_tower` is the
  core pipeline.
- Mission adapters (`demosat_data_utils`, `oco2_seq`, etc

## 3. Exploring Core and Adaptations
TTS is split into **Core Libraries** (shared) and **Mission Adaptations** (mission-specific extensions).

In [3]:
# Core Libraries
!tts-ctx core

# Mission Adaptations
!tts-ctx adaptations

# TTS Core Libraries

# TTS Core Libraries

The `tts_core/` directory contains shared infrastructure libraries. Each is a
standalone pip package with its own repo and Git remote.

## Dependency hierarchy

```
tts_utilities
  ↓
tts_html_utils ──────┬── tts_dictionary_interface
                     └── tts_papertrail
                         ↓
                    tts_data_utils ── tts_dante
                    tts_seq ───────── tts_fresh
                    tts_tower
                    tts_dexter (extends data_utils)
                    tts_spice
                    tts_plan
                    tts_events
                    tts_dtat
                    tts_starter_template
```

## What each library provides

### tts_utilities
Logger (`create_logger`), Python compat helpers (`as_list`, `ensure_dir`). The
base of every other library. No dependencies outside stdlib.

### tts_html_utils
`PowerTable` (tabular HTML generation), `HtmlComponent` component model,
`HtmlCompiler` template engine.

## 4. Global Symbol Search
If you know a keyword or a function name but don't know which repo it's in, use `search`.

In [4]:
!tts-ctx search "filter_expr"

Search: 'filter_expr'

Symbol                                                       | Kind       | Summary
----------------------------------------------------------------------------------------------------
tts_events.core.filter_expr.BoolOp                           | class      | 
tts_events.core.filter_expr.Comparison                       | class      | 
tts_events.core.filter_expr.FieldPath                        | class      | 
tts_events.core.filter_expr.FilterExpressionError            | class      | 
tts_events.core.filter_expr.Literal                          | class      | 
tts_events.core.filter_expr.Not                              | class      | 
tts_events.core.filter_expr._FilterIRTransformer             | class      | 
tts_data_utils.core.data_frame.filter_expr                   | function   | Filter rows where the given boolean expression is True.

Column names refer to columns of this DataFrame. Supports >, >=,
<, <=, ==, !=, is, is not, and the logical combinators 

## 5. Progressive Disclosure (The `class` command)
The `class` command is the most powerful tool for reducing context cost. It lets you zoom in on a class across three levels of detail.

### Depth 1: MRO & Summary
Shows the inheritance chain and a high-level summary. Useful for seeing if a class is a mission adaptation of a core class.

In [5]:
# Example: Oco2Event is a mission-specific adaptation of the core TtsEvent
!tts-ctx class oco2_events.core.event.Oco2Event --depth 1

## oco2_events.core.event.Oco2Event
  "Base class for OCO-2 mission events.

Provides a mission-specific implementation of ``get_chanvals`` that
queries OCO-2 channelized telemetry for this event's time window."
  Repo: 1021
  Public: 1

  MRO:
├── oco2_events.core.event.Oco2Event


### Depth 2: Signatures
Shows the public and private method signatures. This helps you identify *which* method you need to override or call without reading the whole file.

In [6]:
!tts-ctx class tts_data_utils.core.data_frame.TtsDataFrame --depth 2

## tts_data_utils.core.data_frame.TtsDataFrame
  "Base DataFrame subclass for tts_data_utils.

Stores lightweight container-style metadata (``name`` and ``metadata``)
and ensures subclasses are preserved across pandas operations via the
``_constructor`` hook and ``_metadata``."
  Repo: 1227
  Public: 1

  MRO:
├── tts_data_utils.core.data_frame.TtsDataFrame

Methods:


### Depth 3: Source Code
Shows the actual source code lines. This is only used when you are ready to modify the logic.

In [9]:
!tts-ctx class oco2_data_utils.channels.Oco2EvrFrame --depth 3

## oco2_data_utils.channels.Oco2EvrFrame
  ""
  Repo: 994
  Public: 1

  MRO:
├── oco2_data_utils.channels.Oco2EvrFrame

Methods:

Source (/Users/muszynsk/projects/tt_studio/dev/oco2/oco2_data_utils/src/oco2_data_utils/channels.py):
  [Source code snippet would appear here]


## 6. Repository Inspection
Get a summary of a specific repository, including its Git remote and most important classes.

In [ ]:
!tts-ctx repo tts_data_utils